***thematic classification of movie descriptions: Building an Alien/UFO Movie Dataset***

The goal of this notebook is to take the movies dataset and transforming it into a useful dataset for the UFO sightings project. This uses Google Coldab AI to classify and trim the data to only include the movies with extraterrestrial theams.

In [1]:
import os, re, json, time
import pandas as pd
from IPython.display import display
from google.colab import ai, files

SOURCE_URL = 'https://raw.githubusercontent.com/zenith-gravity/ufo-sightings-analysis/refs/heads/main/data/top_rated_movies.csv'
SOURCE_FILE = 'top_rated_movies.csv'
CHECKPOINT = 'alien_movie_ai_results.csv'

# Upload the original CSV if desired; otherwise use the same published source used in your notebook.
if os.path.exists(SOURCE_FILE):
    movies = pd.read_csv(SOURCE_FILE)
else:
    movies = pd.read_csv(SOURCE_URL)

movies = movies.drop(columns=['Unnamed: 0', 'Alien'], errors='ignore')
required = {'id', 'title', 'overview', 'release_date'}
assert required.issubset(movies.columns), f'Missing columns: {required - set(movies.columns)}'
movies['id'] = pd.to_numeric(movies['id'], errors='coerce')
movies = movies.dropna(subset=['id']).copy()
movies['id'] = movies['id'].astype('int64')
movies['title'] = movies['title'].fillna('').astype(str).str.strip()
movies['overview'] = movies['overview'].fillna('').astype(str).str.strip()
raw_rows = len(movies)
movies = movies.drop_duplicates('id', keep='first').copy()
print(f'Source rows: {raw_rows:,}; unique movie IDs: {len(movies):,}; duplicate rows removed: {raw_rows-len(movies):,}')

Source rows: 18,520; unique movie IDs: 9,959; duplicate rows removed: 8,561


Search titles and plot overviews for exteraterrestial terms. This is a an intial sorting step. A term such as "invasion" may refer to human war, but should be set aside for a human review.

In [2]:
PATTERN = (r'\b(?:aliens?|extraterrestrials?|ufos?|unidentified flying objects?|'
           r'flying saucers?|spaceships?|spacecraft|martians?|xenomorphs?|'
           r'interstellar|invasion|abduction|first contact|other planets?|outer space)\b')
searchable = movies['title'] + ' ' + movies['overview']
candidates = movies.loc[searchable.str.contains(PATTERN, case=False, regex=True)].copy()
candidates = candidates.drop_duplicates('id')
print(f'Unique keyword candidates: {len(candidates):,}')
display(candidates[['id', 'title', 'release_date']].head(10))

Unique keyword candidates: 228


,id,title,release_date
14,157336,Interstellar,2014-11-05
108,25237,Come and See,1985-10-17
133,348,Alien,1979-05-25
218,1091,The Thing,1982-06-25
254,550776,Believe Me: The Abduction of Lisa McVey,2018-09-30
339,679,Aliens,1986-07-18
433,94047,My Way,2011-12-21
542,47931,Elite Squad: The Enemy Within,2010-10-08
603,398924,A Bag of Marbles,2017-01-18
617,11049,Interstella 5555: The 5tory of the 5ecret 5tar...,2003-05-28


This checkpoint files and stores earlier AI decisions so we can pick up where we left off instead of paying the compution cost each time.

In [3]:
# Set True only if your checkpoint needs to be uploaded into a fresh Colab session.
UPLOAD_EXISTING_CHECKPOINT = False
if UPLOAD_EXISTING_CHECKPOINT:
    files.upload()  # select alien_movie_ai_results.csv

expected_cols = ['id','label','category','reason']
if os.path.exists(CHECKPOINT):
    labels = pd.read_csv(CHECKPOINT)
    assert set(expected_cols).issubset(labels.columns), 'Checkpoint is missing expected columns'
    labels = labels[expected_cols].copy()
else:
    labels = pd.DataFrame(columns=expected_cols)

labels['id'] = pd.to_numeric(labels['id'], errors='coerce')
labels = labels.dropna(subset=['id']).copy()
labels['id'] = labels['id'].astype('int64')
labels['label'] = labels['label'].fillna('').astype(str).str.upper().str.strip()
labels['category'] = labels['category'].fillna('').astype(str).str.strip()
labels['reason'] = labels['reason'].fillna('').astype(str).str.strip()
labels = labels.drop_duplicates('id', keep='last')
labels = labels.loc[labels['label'].isin(['YES', 'NO', 'UNCERTAIN'])]
print('Valid AI classifications loaded:', len(labels))
print('Unique candidates still missing:', (~candidates['id'].isin(labels['id'])).sum())

Valid AI classifications loaded: 0
Unique candidates still missing: 228


Give the model perdefined categories (alien_life, ufo_sighting, alien_invasion, etc) and ask for yes no or uncertain, plus a short explanation. small batches are vaildated and saved as we go.

This way we can get rid of the no, and review the uncertain.

In [4]:
def make_prompt(batch):
    records = [{'id':int(r.id), 'title':r.title, 'overview':r.overview[:850]}
               for r in batch.itertuples()]
    return (
        'Classify movie titles and overviews. Return ONLY a JSON array, one object per input id. '
        'Each object must have id (integer), label (YES, NO, UNCERTAIN), '
        'category (alien_life, ufo_sighting, alien_invasion, alien_abduction, first_contact, '
        'other_extraterrestrial, none, uncertain), and a short reason. '
        'YES: extraterrestrial beings/intelligence, UFOs, alien contact/invasion/abduction '
        'are meaningful parts of the story. NO: ordinary space travel without alien life, '
        'human invasions or unrelated use of alien/abduction. UNCERTAIN if inadequate evidence. '
        'Use only supplied information; do not invent plot details. Input JSON:\n'
        + json.dumps(records, ensure_ascii=False)
    )

def parse_response(answer):
    raw = str(answer).strip()
    raw = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw, flags=re.I).strip()
    result = json.loads(raw)
    if not isinstance(result, list):
        raise ValueError('AI response was not a JSON list')
    return result

BATCH_SIZE = 8
MAX_BATCHES = None  # None = process every missing unique candidate
pending = candidates.loc[~candidates['id'].isin(labels['id'])]
chunks = [pending.iloc[i:i+BATCH_SIZE] for i in range(0, len(pending), BATCH_SIZE)]
if MAX_BATCHES is not None:
    chunks = chunks[:MAX_BATCHES]

for i, batch in enumerate(chunks, 1):
    try:
        response = ai.generate_text(make_prompt(batch))
        new = pd.DataFrame(parse_response(response))
        if not set(['id','label','category','reason']).issubset(new.columns):
            raise ValueError('AI response missing required fields')
        new = new[['id','label','category','reason']].copy()
        new['id'] = pd.to_numeric(new['id'], errors='raise').astype('int64')
        new['label'] = new['label'].astype(str).str.upper().str.strip()
        if len(new) != len(batch) or new['id'].duplicated().any() or set(new['id']) != set(batch['id']):
            raise ValueError('AI returned missing, repeated, or extra IDs')
        if not new['label'].isin(['YES','NO','UNCERTAIN']).all():
            raise ValueError('Invalid classification labels')
        labels = pd.concat([labels, new], ignore_index=True).drop_duplicates('id', keep='last')
        labels.to_csv(CHECKPOINT, index=False)
        print(f'Batch {i}/{len(chunks)} saved; total valid labels: {len(labels)}')
    except Exception as exc:
        print(f'Batch {i} failed: {exc}; previous results are safe in {CHECKPOINT}.')
        break

print('Unique candidate count:', len(candidates))
print('Unique classified candidates:', candidates['id'].isin(labels['id']).sum())
print('Unique candidates not yet classified:', (~candidates['id'].isin(labels['id'])).sum())

Batch 1/29 saved; total valid labels: 8
Batch 2/29 saved; total valid labels: 16
Batch 3/29 saved; total valid labels: 24
Batch 4/29 saved; total valid labels: 32
Batch 5/29 saved; total valid labels: 40
Batch 6/29 saved; total valid labels: 48
Batch 7/29 saved; total valid labels: 56
Batch 8/29 saved; total valid labels: 64
Batch 9/29 saved; total valid labels: 72
Batch 10/29 saved; total valid labels: 80
Batch 11/29 saved; total valid labels: 88
Batch 12/29 saved; total valid labels: 96
Batch 13/29 saved; total valid labels: 104
Batch 14/29 saved; total valid labels: 112
Batch 15/29 saved; total valid labels: 120
Batch 16/29 saved; total valid labels: 128
Batch 17/29 saved; total valid labels: 136
Batch 18/29 saved; total valid labels: 144
Batch 19/29 saved; total valid labels: 152
Batch 20/29 saved; total valid labels: 160
Batch 21/29 saved; total valid labels: 168
Batch 22/29 saved; total valid labels: 176
Batch 23/29 saved; total valid labels: 184
Batch 24/29 saved; total valid la

Convert movie release dates to the same YYYY-MM-DD calendar format we will eventually use with UFO sighting dates. This first export contains only AI YES films with valid dates; uncertain ones go to a review file. Important: the final approved export comes later, so the file written in this section is an intermediate version.

In [5]:
classified = candidates.merge(labels, on='id', how='inner', validate='one_to_one')
classified['release_date'] = pd.to_datetime(classified['release_date'], errors='coerce')
classified['release_date'] = classified['release_date'].dt.strftime('%Y-%m-%d')
classified['category'] = classified['category'].replace('', 'uncertain')

main_columns = ['id','title','release_date','label','category','reason','overview']
confirmed = (classified.loc[(classified['label']=='YES') & classified['release_date'].notna(), main_columns]
             .drop_duplicates('id').sort_values(['release_date','title']).reset_index(drop=True))
review = (classified.loc[(classified['label']=='UNCERTAIN') |
                         ((classified['label']=='YES') & classified['release_date'].isna()), main_columns]
          .drop_duplicates('id').reset_index(drop=True))
missing = candidates.loc[~candidates['id'].isin(labels['id']), ['id','title','release_date','overview']]

confirmed.to_csv('alien_ufo_movies_clean.csv', index=False)
review.to_csv('alien_ufo_movies_review.csv', index=False)
missing.to_csv('alien_ufo_movies_not_classified.csv', index=False)
print('Unique movie candidates:',len(candidates))
print('Classified:',len(classified))
print('YES with valid dates (clean export):', len(confirmed))
print('Uncertain or YES with missing/bad dates:', len(review))
print('Not yet classified:',len(missing))
print('Label counts:\n', classified['label'].value_counts(dropna=False).to_string())
display(confirmed.head(15))

Unique movie candidates: 228
Classified: 228
YES with valid dates (clean export): 163
Uncertain or YES with missing/bad dates: 13
Not yet classified: 0
Label counts:
 label
YES          163
NO            52
UNCERTAIN     13


,id,title,release_date,label,category,reason,overview
0,10785,The Thing from Another World,1951-04-05,YES,alien_life,Scientists and military officials fight agains...,Scientists and US Air Force officials fend off...
1,828,The Day the Earth Stood Still,1951-09-18,YES,first_contact,An alien and a robot arrive on Earth to delive...,An alien and a robot land on Earth after World...
2,11549,Invasion of the Body Snatchers,1956-02-05,YES,alien_invasion,The plot centers on a community being covertly...,A small-town doctor learns that the population...
3,8851,The Blob,1958-09-10,YES,alien_life,The story centers on a gelatinous alien life f...,"A drive-in favorite, this sci-fi classic follo..."
4,10514,The Andromeda Strain,1971-03-12,YES,alien_life,"Scientists isolate and study a dangerous, muta...",When virtually all of the residents of Piedmon...
5,1410,Dark Star,1974-03-30,YES,alien_life,The crew must battle an alien mascot aboard th...,A group of scientists are sent on a mission to...
6,991,The Man Who Fell to Earth,1976-03-18,YES,alien_life,The protagonist is an extraterrestrial who tra...,Thomas Jerome Newton is an alien who has come ...
7,840,Close Encounters of the Third Kind,1977-12-14,YES,ufo_sighting,The plot centers on a protagonist who has an e...,"After an encounter with UFOs, an electricity l..."
8,10516,The Gendarme and the Creatures from Outer Space,1979-01-31,YES,alien_invasion,"Humanoid, oil-drinking aliens from outer space...",The bungling inspector Cruchot finds himself t...
9,348,Alien,1979-05-25,YES,alien_life,The plot centers on an encounter with an alien...,"During its return to the earth, commercial spa..."


uploaded the results and reviewed the uncertian ones

In [7]:
#files.download('alien_ufo_movies_clean.csv')
#files.download('alien_ufo_movies_review.csv')
# If needed: files.download('alien_movie_ai_results.csv')
# If needed: files.download('alien_ufo_movies_not_classified.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

After reviewing the uncertain movies and chose to keep them. In this section we add those records to the YES movies and preserve the AI’s original label for transparency. We intentionally keep the AI’s category and reason unchanged; a previously uncertain category may still need manual refinement for future category-level analysis.

In [8]:
# Get all confirmed and previously uncertain movies
approved = classified[
    classified['label'].isin(['YES', 'UNCERTAIN'])
].copy()

# Preserve original classification
approved['original_label'] = approved['label']

# You reviewed and approved the uncertain movies
approved['label'] = 'YES'

# Merge movie details, including release dates
movie_details = movies.drop_duplicates(subset='id')

final_movies = movie_details.merge(
    approved[['id', 'label', 'category',
              'reason', 'original_label']],
    on='id',
    how='inner'
)

# Standardize release dates
final_movies['release_date'] = pd.to_datetime(
    final_movies['release_date'],
    errors='coerce'
)

# Keep movies with valid dates
final_movies = final_movies.dropna(subset=['release_date'])

# Remove duplicate movies
final_movies = final_movies.drop_duplicates(subset='id')

# Select final columns
final_movies = final_movies[
    ['id', 'title', 'release_date',
     'label', 'category', 'reason', 'original_label']
].sort_values('release_date')

# Format dates
final_movies['release_date'] = (
    final_movies['release_date'].dt.strftime('%Y-%m-%d')
)

# Export the completed dataset
final_movies.to_csv('alien_ufo_movies_clean.csv', index=False)

print("Final alien/UFO movies:", len(final_movies))
display(final_movies.head(20))

Final alien/UFO movies: 176


,id,title,release_date,label,category,reason,original_label
70,10785,The Thing from Another World,1951-04-05,YES,alien_life,Scientists and military officials fight agains...,YES
18,828,The Day the Earth Stood Still,1951-09-18,YES,first_contact,An alien and a robot arrive on Earth to delive...,YES
15,11549,Invasion of the Body Snatchers,1956-02-05,YES,alien_invasion,The plot centers on a community being covertly...,YES
30,830,Forbidden Planet,1956-03-02,YES,uncertain,Mentions space travel to another planet and an...,UNCERTAIN
117,8851,The Blob,1958-09-10,YES,alien_life,The story centers on a gelatinous alien life f...,YES
44,10514,The Andromeda Strain,1971-03-12,YES,alien_life,"Scientists isolate and study a dangerous, muta...",YES
141,1410,Dark Star,1974-03-30,YES,alien_life,The crew must battle an alien mascot aboard th...,YES
98,991,The Man Who Fell to Earth,1976-03-18,YES,alien_life,The protagonist is an extraterrestrial who tra...,YES
23,840,Close Encounters of the Third Kind,1977-12-14,YES,ufo_sighting,The plot centers on a protagonist who has an e...,YES
32,11850,Invasion of the Body Snatchers,1978-12-20,YES,uncertain,Describes residents becoming drone-like shadow...,UNCERTAIN


In [13]:
# Ensure release_date is parsed as datetime in final_movies
release_dates = pd.to_datetime(final_movies['release_date'], errors='coerce')

# Extract month and year as new columns directly in final_movies
final_movies['year'] = release_dates.dt.year
final_movies['month'] = release_dates.dt.month

# Save final_movies back to the CSV file
final_movies.to_csv('alien_ufo_movies_clean.csv', index=False)

# Display the first few rows to confirm
print("Columns added to final_movies dataset successfully!")
display(final_movies.head())


Columns added to final_movies dataset successfully!


,id,title,release_date,label,category,reason,original_label,year,month
70,10785,The Thing from Another World,1951-04-05,YES,alien_life,Scientists and military officials fight agains...,YES,1951,4
18,828,The Day the Earth Stood Still,1951-09-18,YES,first_contact,An alien and a robot arrive on Earth to delive...,YES,1951,9
15,11549,Invasion of the Body Snatchers,1956-02-05,YES,alien_invasion,The plot centers on a community being covertly...,YES,1956,2
30,830,Forbidden Planet,1956-03-02,YES,uncertain,Mentions space travel to another planet and an...,UNCERTAIN,1956,3
117,8851,The Blob,1958-09-10,YES,alien_life,The story centers on a gelatinous alien life f...,YES,1958,9


In [15]:
from google.colab import files

final_movies.to_csv('alien_ufo_movies_clean.csv', index=False)
files.download('alien_ufo_movies_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>